## About this notebook:

This notebook finds how each round is shifted and rotated relative to the anchor round (the first round, unless `experiment.json` sets `anchor_round`), by aligning the round's cell outlines from 01 onto the anchor's with StackReg. Several rounds run at once.

Input:
- `segmented/{well}/`: every round's cell outlines from 01
- `output_df/round_table_{well}.csv` from 00

Settings:
- `downscale_factor`: 1 aligns the outlines at full size; 2 aligns them at half the width and height (4 times fewer pixels), for montages too large for the computer's memory.
- `overwrite`: `False` (the default) aligns only rounds that have no saved alignment yet and keeps every other round, including rounds fixed by hand in 03. `True` aligns every round again and replaces those fixes. If the rounds or the anchor round have changed since the alignments were saved, it stops and asks for `overwrite = True`.

Output:
- `output_df/tmat_{well}.npy`: one transform per round, used by 03 and 06.

The last cells (optional) open the alignment viewer to check the result, the same viewer as 03.

## Choose the experiment

Set `CONFIG` to the experiment's `experiment.json`. All folders and the well list come from it; create or edit it with `setup_experiment.ipynb`.

In [ ]:
from fouri import load_config

# This experiment's experiment.json (create or edit it with setup_experiment.ipynb)
CONFIG = r'Z:\PATH\TO\EXPERIMENT\experiment.json'

exp = load_config(CONFIG)

# specify a downscaling factor for the alignment
downscale_factor = 2
overwrite = False  # True aligns every round again, replacing rounds already aligned, including fixes from 03

## Prepare for processing

In [ ]:
from fouri.align import find_transforms

## Process the wells

In [ ]:
transforms = find_transforms(exp, downscale_factor=downscale_factor, n_workers=None, overwrite=overwrite)

## Optional - check the alignment in napari

Opens the alignment viewer (the same as 03) on a well, with every round where 02 put it. A round that needs fixing can be fixed right there.

* Magenta = anchor round, green = the round shown, white = cells that line up.
* **Move around:** pick a well or round in the panel, or use the arrows there. On the keyboard, Up / Down change the round and Left / Right the well; with Shift they jump to the first or last. Scroll to zoom; hold Space and drag to pan.
* **Start from:** *Saved alignment* shows each round where its saved transform puts it (a round without one starts unaligned); *No alignment* shows it where it sits in its own image, to align it from scratch.
* **Fixing a round:** click a cell in magenta, then the same cell in green. Click anywhere inside the cell: the dot jumps to its centre, and a click outside any cell is ignored. Repeat for at least 3 cells, two of them far apart (a third of the image or more), then press **Fit (Shift+F)**. Add pairs and fit again to improve it.
* **Undo (Ctrl+Z)** removes your last click; **Save (Shift+S)** replaces this round's transform for the well. Switching round or well forgets unsaved clicks.


In [ ]:
from fouri.viewers import AlignmentViewer

well = exp.well_list[0]  # the well to open first
viewer = AlignmentViewer(exp, well)